# Data Cleaning and Preprocessing

This notebook contains the necessary code to clean the products dataset and perform any necessary preprocessing. The first cells below consist of data loading and data types assignation, followed by cells dedicated to tasks such as column dropping, value inference or standarization.

## Load data

In [181]:
import pandas as pd
pd.set_option('display.max_columns', None)

# Flag used to determine if data is used to generate the limited csv up until february 2026 or if it is limited to the last entry available
# false = dataset limted to February 2026
# true  = dataset limited to last data entry date (used in AB testing section)
#
# Note: This part is only done in this section as Product specific data is the only data that was used for an extended period of time
#
use_full_dataset = False
# use_full_dataset = True

In [182]:
df_products = pd.read_csv("../../Datasets/RawDatasets/Productes/ProductesMargins.csv", dtype = "str")
df_products.head()

,Comanda,Producte,Categoria,Quantitat,Marge,margin_percentage,Pes,Tipologia,Atributs,ID de producte,Extra ID,Codi de barres,Tipus,Format,Taula,Treballador,Tarifa,Grup analític,Preu,Sub-total,Percentatge de descompte,Descompte,Impost total,Impost,Total,Data,Tipus.1,Descompte percentual
0,1,AQUARIUS TARONJA,REFRESC,1,1.55,79.59,--,Estàndard,0,136,44285396,NaN,Estàndard,--,--,--,--,--,1.95,"1,77",NaN,0,"0,18",--,1.95,2018-08-03 17:47:45,NaN,NaN
1,1,VARIS CAFE,CAFES+PASTES+INF.,1,1.32,71.46,--,Estàndard,NaN,640,NaN,NaN,Estàndard,--,--,--,--,--,1.85,"1,68",NaN,0,"0,17",--,1.85,2018-08-03 17:47:45,NaN,NaN
2,2,MARZEN POSTE BARRIL,CERVESA,1,1.23,62.96,--,Estàndard,NaN,164,8942475,NaN,Estàndard,--,--,--,--,--,1.95,"1,77",NaN,0,"0,18",--,1.95,2018-08-03 17:48:25,NaN,NaN
3,3,"POSTE 33cl 16 unit,",CERVESA,3,2.15,44.83,--,Estàndard,0,157,11035215,NaN,Estàndard,--,--,--,--,--,1.6,"4,36",NaN,0,"0,44",--,4.8,2018-08-03 17:59:42,NaN,NaN
4,5,LADRON DE MANZANAS,CERVESA,2,3.04,67.51,--,Estàndard,NaN,162,44296142,NaN,Estàndard,--,--,--,--,--,2.25,"4,09",NaN,0,"0,41",--,4.5,2018-08-03 18:13:32,NaN,NaN


In [183]:
# Remove leading and trailing spaces
df_products = df_products.apply(lambda col: col.str.lstrip() if col.dtype == 'object' else col)

In [184]:
df_products['Quantitat'] = pd.to_numeric(df_products['Quantitat'], errors='coerce').astype(int)
df_products['margin_percentage'] = pd.to_numeric(df_products['margin_percentage'].str.replace(',', '.'), errors='coerce')
df_products['Marge'] = pd.to_numeric(df_products['Marge'].str.replace(',', '.'), errors='coerce')
df_products['Pes'] = pd.to_numeric(df_products['Pes'].str.replace(',', '.'), errors='coerce')
df_products['Atributs'] = pd.to_numeric(df_products['Atributs'], errors='coerce').fillna(0).astype(int)
df_products['ID de producte'] = pd.to_numeric(df_products['ID de producte'], errors='coerce').astype(int)
df_products['Extra ID'] = pd.to_numeric(df_products['Extra ID'], errors='coerce').fillna(0).astype(int)
df_products['Preu'] = pd.to_numeric(df_products['Preu'].str.replace(',', '.'), errors='coerce')
df_products['Sub-total'] = pd.to_numeric(df_products['Sub-total'].str.replace(',', '.'), errors='coerce')
df_products['Percentatge de descompte'] = pd.to_numeric(df_products['Percentatge de descompte'].str.replace(',', '.'), errors='coerce')
df_products['Descompte'] = pd.to_numeric(df_products['Descompte'].str.replace(',', '.'), errors='coerce')
df_products['Impost total'] = pd.to_numeric(df_products['Impost total'].str.replace(',', '.'), errors='coerce')
df_products['Impost'] = pd.to_numeric(df_products['Impost'].str.replace(',', '.'), errors='coerce')
df_products['Total'] = pd.to_numeric(df_products['Total'].str.replace(',', '.'), errors='coerce')
df_products['Data'] = pd.to_datetime(df_products['Data'])

df_products.dtypes

Comanda                             object
Producte                            object
Categoria                           object
Quantitat                            int64
Marge                              float64
margin_percentage                  float64
Pes                                float64
Tipologia                           object
Atributs                             int64
ID de producte                       int64
Extra ID                             int64
Codi de barres                      object
Tipus                               object
Format                              object
Taula                               object
Treballador                         object
Tarifa                              object
Grup analític                       object
Preu                               float64
Sub-total                          float64
Percentatge de descompte           float64
Descompte                          float64
Impost total                       float64
Impost     

In [185]:
# Crop data up until February 2026 for EDA and MBA stages. Alternatively, leave as is for AB testing phase.
if not use_full_dataset:
    df_products = df_products[df_products['Data'] < '2026-03-01'].copy()

print("Date range:")
print(f"From: {df_products['Data'].min()}")
print(f"To:   {df_products['Data'].max()}")

Date range:
From: 2018-08-03 17:47:45
To:   2026-02-28 16:19:16


## Missing Order ID
It has been detected that some of the different products, from 2018 to 2023, do not pertain to any order. In depth exploration revealed that they are almost all products with a price tag of 0€ and normally pertain to a menu. Later years, probably from a system restructure, non-priced products inside a menu have an order id equal to the menu's order id.

Products with a higher price tag have been identified as a add-ons to a menu. As an example, sodas in the daily menu are not included but can be added with an increase in the menu's pricing of just 1€. This products have been flagged as possibly interesting for later analysis and stored in a separate dataset.

In [187]:
invalid_comanda_rows = df_products[pd.to_numeric(df_products['Comanda'], errors='coerce').isna()]
category_counts = invalid_comanda_rows['Categoria'].value_counts()
total_invalid = len(invalid_comanda_rows)
total = len(df_products)

print("Counts of invalid 'Comanda' per category:")
print(category_counts)
print(f"\nTotal invalid 'Comanda' entries: {total_invalid}")
print(f"Percentage of invalid data: {total_invalid / total * 100:.2f}%")

Counts of invalid 'Comanda' per category:
Categoria
PLATS MENU - PRIMERS    62125
PLATS MENU - SEGONS     60185
PLATS MENU - POSTRES    43445
POSTRES                 15458
REFRESC                  9440
PRIMERS                  5407
CARN                     3561
PEIX                     2722
CERVESA                  1884
BEGUDES MENU             1557
PLATS MENU INFANTIL      1548
CAFES+PASTES+INF.        1064
GELATS                    862
SUPLEMENTS                141
VINS                       65
ARROSOS                     9
ESMORZARS                   6
TAPES                       4
Name: count, dtype: int64

Total invalid 'Comanda' entries: 209483
Percentage of invalid data: 17.59%


In [188]:
menu_invalid = invalid_comanda_rows[invalid_comanda_rows['Preu'] == 0]
non_menu_invalid = invalid_comanda_rows[invalid_comanda_rows['Preu'] > 0]

print(f"Price-0 invalid rows (safe to drop): {len(menu_invalid)} ({len(menu_invalid) / total * 100:.2f}%)")
print(f"Priced invalid rows (worth keeping): {len(non_menu_invalid)} ({len(non_menu_invalid) / total * 100:.2f}%)")

Price-0 invalid rows (safe to drop): 204547 (17.17%)
Priced invalid rows (worth keeping): 4936 (0.41%)


In [189]:
# Save priced invalid rows to separate dataset
df_orphaned = non_menu_invalid.copy()
df_orphaned.to_csv("../../Datasets/CleanedDatasets/DirtyMenuItems.csv", index=False)

# Drop all invalid Comanda rows from main dataset
df_products = df_products[pd.to_numeric(df_products['Comanda'], errors='coerce').notna()]

print(f"Rows remaining in main dataset: {len(df_products)}")
print(f"Rows in orphaned dataset: {len(df_orphaned)}")

Rows remaining in main dataset: 981644
Rows in orphaned dataset: 4936


In [190]:
df_products['Comanda'] = pd.to_numeric(df_products['Comanda'], errors='coerce').astype(int)

## Drop unnecessary columns (Nan)
The dataset contains certain columns with no data in them. The main reason for this observation comes from the fact that the restaurant uses a service widely utilized in the sector, the datasets are not tailored to this specific restaurant needs and some of the features offered by the Revo service are left unused or unconfigured. Since this columns provide no value and will only slow down performance, they have been opted to be removed.

In [192]:
missing = pd.DataFrame({
    'count': df_products.isna().sum(),
    'percentage': df_products.isna().mean() * 100
})
missing[missing["count"] > 0].head()

,count,percentage
Pes,981644,100.000000
Tipologia,3726,0.379567
Codi de barres,981644,100.000000
Percentatge de descompte,981595,99.995008
Impost,981644,100.000000


The above exploration of the dataset shows that there are four columns with missing data, with three of those having a 100% of missing data in them. First of all, we will start by removing the columns with absolutly no information in them and explore deeper into the discount percentage column.

In [194]:
columns_to_drop = ['Pes', 'Codi de barres', 'Impost', 'Tipus.1']
df_products = df_products.drop(columns = columns_to_drop)

In [195]:
df_products["Percentatge de descompte"].value_counts()

Percentatge de descompte
100.0    49
Name: count, dtype: int64

In the previous cell we can observe how the only discount percentage is a 100%, this corresponds to clients that for certain reasons have been invited (e.g. promotion). 

The main issue with this is that when the restaurant workers provides a discount for someone they tend to delete the entire order instead of applying a discount percentage to the products and completing the order (it is an easier and faster way of obtaining the same results), therefore analyzing this data may result in biased conclusions and the chosen option to treat this column has been to drop it.

Since Descompte is a directly correlated column, it will also be droped as it gives the same information in a monetary value instead of a percentage.

In [197]:
df_products = df_products.drop(columns = ["Percentatge de descompte", "Descompte", "Descompte percentual"])

df_products.head()

,Comanda,Producte,Categoria,Quantitat,Marge,margin_percentage,Tipologia,Atributs,ID de producte,Extra ID,Tipus,Format,Taula,Treballador,Tarifa,Grup analític,Preu,Sub-total,Impost total,Total,Data
0,1,AQUARIUS TARONJA,REFRESC,1,1.55,79.59,Estàndard,0,136,44285396,Estàndard,--,--,--,--,--,1.95,1.77,0.18,1.95,2018-08-03 17:47:45
1,1,VARIS CAFE,CAFES+PASTES+INF.,1,1.32,71.46,Estàndard,0,640,0,Estàndard,--,--,--,--,--,1.85,1.68,0.17,1.85,2018-08-03 17:47:45
2,2,MARZEN POSTE BARRIL,CERVESA,1,1.23,62.96,Estàndard,0,164,8942475,Estàndard,--,--,--,--,--,1.95,1.77,0.18,1.95,2018-08-03 17:48:25
3,3,"POSTE 33cl 16 unit,",CERVESA,3,2.15,44.83,Estàndard,0,157,11035215,Estàndard,--,--,--,--,--,1.60,4.36,0.44,4.80,2018-08-03 17:59:42
4,5,LADRON DE MANZANAS,CERVESA,2,3.04,67.51,Estàndard,0,162,44296142,Estàndard,--,--,--,--,--,2.25,4.09,0.41,4.50,2018-08-03 18:13:32


## Drop unnecessary columns (Missing data)
The previous cells dropped columns that were missing information, in the following cells the columns explored might also be missing information in other forms (e.g. Zero values instead of Nan) and will be inspected in order to decide how to proceed with them.

In [199]:
df_products["Atributs"].value_counts()

Atributs
0    981644
Name: count, dtype: int64

In [200]:
df_products["Tipus"].value_counts()

Tipus
Estàndard          981294
Menú                  322
Format de venda        28
Name: count, dtype: int64

In [201]:
df_products["Format"].value_counts()

Format
--          969186
XUPITO        4596
TUB           3756
COMBINAT      2104
COPA          1980
AMPOLLA         22
Name: count, dtype: int64

In [202]:
df_products["Treballador"].value_counts()

Treballador
Revo Systems    897585
--               80742
InTouch           2368
JORDI              891
JANIRA              57
SOLO                 1
Name: count, dtype: int64

In [203]:
df_products["Tarifa"].value_counts()

Tarifa
--          960000
Terrassa     21624
Standard        20
Name: count, dtype: int64

In [204]:
df_products["Grup analític "].value_counts()

Grup analític 
--    981644
Name: count, dtype: int64

#### Analysis
From the previous analysis of the data we can extract the following information about the columns from which we previously had no data on their distribution.

- **Atributs**: This column represents attached attributes to a given product, it is a feature the restaurant does not utilize and therefore holds no significant value. This column will be __dropped__.
- **Tipus**: Another column representing data that the restaurant has not configured. The type of the product is always standard and therefore adds no value to further analysis. This column will be __dropped__.
- **Format**: This columns depicts the format in which a licor (the only category the restaurant has configured with this feature) can be served. This has the potential to provide information and will __not be dropped__.
- **Treballador**: This column represents the worker that has taken the order, the restaurant seemed to use this feature during a really short amount of time but ended up using a common woker (Revo Systems) for all transactions. Since the tablet in which the orders are made is shared among workers it was not convinient to keep switching. Since this column may bias results (e.g. Jordi, the owner, only appears on 888 instances), the column will be __dropped__.
- **Tarifa**: This is a recently used column. The restaurant has a large outdoor space to which clients can go with two possibilities on how to be served, they can either order inside and take everything out themselves or they can be attended fully outside with a surplus in their total. This column will __not be dropped__.
- **Grup analític**: This is an unused feature and therefore will be __dropped__.

In [206]:
columns_to_drop = ['Atributs', 'Tipus', 'Treballador', 'Grup analític ']
df_products = df_products.drop(columns = columns_to_drop)

df_products.head()

,Comanda,Producte,Categoria,Quantitat,Marge,margin_percentage,Tipologia,ID de producte,Extra ID,Format,Taula,Tarifa,Preu,Sub-total,Impost total,Total,Data
0,1,AQUARIUS TARONJA,REFRESC,1,1.55,79.59,Estàndard,136,44285396,--,--,--,1.95,1.77,0.18,1.95,2018-08-03 17:47:45
1,1,VARIS CAFE,CAFES+PASTES+INF.,1,1.32,71.46,Estàndard,640,0,--,--,--,1.85,1.68,0.17,1.85,2018-08-03 17:47:45
2,2,MARZEN POSTE BARRIL,CERVESA,1,1.23,62.96,Estàndard,164,8942475,--,--,--,1.95,1.77,0.18,1.95,2018-08-03 17:48:25
3,3,"POSTE 33cl 16 unit,",CERVESA,3,2.15,44.83,Estàndard,157,11035215,--,--,--,1.60,4.36,0.44,4.80,2018-08-03 17:59:42
4,5,LADRON DE MANZANAS,CERVESA,2,3.04,67.51,Estàndard,162,44296142,--,--,--,2.25,4.09,0.41,4.50,2018-08-03 18:13:32


## Columns renaming
The information in the dataset consists of data from a restaurant in Catalonia and therefore, column and categorical data are often in catalan. The name of products will not be changed but in order to align the language of the data with the language of the final thesis all column name will be translated. Columns have also been translated to snake_case to facilitate programming flows.

In [208]:
catalan_to_english = [
    "order",
    "product",
    "category",
    "quantity",
    "margin",
    "margin_percentage",
    "type",
    "product_id",
    "extra_id",
    "format",
    "table",
    "fee",
    "price",
    "subtotal",
    "total_taxes",
    "total",
    "date"
]

# Assign the new list to the columns attribute
df_products.columns = catalan_to_english

df_products.head()

,order,product,category,quantity,margin,margin_percentage,type,product_id,extra_id,format,table,fee,price,subtotal,total_taxes,total,date
0,1,AQUARIUS TARONJA,REFRESC,1,1.55,79.59,Estàndard,136,44285396,--,--,--,1.95,1.77,0.18,1.95,2018-08-03 17:47:45
1,1,VARIS CAFE,CAFES+PASTES+INF.,1,1.32,71.46,Estàndard,640,0,--,--,--,1.85,1.68,0.17,1.85,2018-08-03 17:47:45
2,2,MARZEN POSTE BARRIL,CERVESA,1,1.23,62.96,Estàndard,164,8942475,--,--,--,1.95,1.77,0.18,1.95,2018-08-03 17:48:25
3,3,"POSTE 33cl 16 unit,",CERVESA,3,2.15,44.83,Estàndard,157,11035215,--,--,--,1.60,4.36,0.44,4.80,2018-08-03 17:59:42
4,5,LADRON DE MANZANAS,CERVESA,2,3.04,67.51,Estàndard,162,44296142,--,--,--,2.25,4.09,0.41,4.50,2018-08-03 18:13:32


## Save cleaned data
Now that data has been cleaned, processed and is ready for further analysis, it will be saved to a new csv.

In [210]:
if not use_full_dataset:
    df_products.to_csv("../../Datasets/CleanedDatasets/ProductsClean.csv", index = False)
else:
    df_products.to_csv("../../Datasets/CleanedDatasets/ProductsCleanBaseline.csv", index = False)